In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
with open('bomex_25m_r20251009/pkl/csd_stats.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open('bomex_25m_ehe18_r20251009/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe18 = pickle.load(f)
with open('bomex_25m_ehe21_r20251107/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe21 = pickle.load(f)
with open('bomex_25m_ehe22_r20251107/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe22 = pickle.load(f)

In [ ]:
nc_ctl, = stat_ctl[0].shape
nc_ehe18, = stat_ehe18[0].shape
nc_ehe21, = stat_ehe21[0].shape
nc_ehe22, = stat_ehe22[0].shape
print(f'{nc_ctl} clouds in CTL simulation')
print(f'{nc_ehe18} clouds in EHE18 simulation')
print(f'{nc_ehe21} clouds in EHE21 simulation')
print(f'{nc_ehe22} clouds in EHE22 simulation')

## Compare the Distributions

In [ ]:
dz = 25 # m
z = np.arange(dz/2, 3000., dz)

In [ ]:
minmf = 0
maxmf = np.max([np.max(stat_ehe18[0]), np.max(stat_ehe21[0]), np.max(stat_ehe22[0]), np.max(stat_ctl[0])]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
def compare_csd_norm(stat_ehe, stat_ctl, name_ehe, nbins, minmf, maxmf):
    bins = np.linspace(minmf, maxmf, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_widths = np.diff(bins)

    # Use only positive, finite values before log10
    ehe_raw = np.asarray(stat_ehe[0], dtype=float)
    ctl_raw = np.asarray(stat_ctl[0], dtype=float)
    ehe_log = np.log10(ehe_raw[(ehe_raw > 0) & np.isfinite(ehe_raw)])
    ctl_log = np.log10(ctl_raw[(ctl_raw > 0) & np.isfinite(ctl_raw)])

    # Normalized histogram (fraction per bin)
    h_ehe, _ = np.histogram(ehe_log, bins=bins)
    h_ctl, _ = np.histogram(ctl_log, bins=bins)

    h_ehe = h_ehe / h_ehe.sum()
    h_ctl = h_ctl / h_ctl.sum()

    # Convert to %
    h_ehe_pct = 100.0 * h_ehe
    h_ctl_pct = 100.0 * h_ctl
    diff_pct = h_ehe_pct - h_ctl_pct
    print(diff_pct)

    fig = plt.figure(figsize=(12, 14))
    ax1 = fig.add_axes((0.1, 0.55, 0.85, 0.4))
    ax2 = fig.add_axes((0.1, 0.1, 0.85, 0.4))

    # Panel 1: normalized distributions
    ax1.bar(bin_centers, h_ctl_pct, width=bin_widths, color='black', alpha=0.45, label='CTL')
    ax1.bar(bin_centers, h_ehe_pct, width=bin_widths, color='red', alpha=0.45, label=name_ehe)
    ax1.axvline(np.median(ehe_log), color='red', linestyle='--')
    ax1.axvline(np.median(ctl_log), color='black', linestyle='--')
    ax1.set_yscale('log')
    ax1.set_ylabel('Cloud fraction per bin (%)')
    ax1.set_title('Normalized histograms')
    ax1.legend()

    ax2.bar(bin_centers, np.where(diff_pct>0.0, diff_pct, np.nan), width=bin_widths, color='red', label=rf'{name_ehe} $>$ CTL')
    ax2.bar(bin_centers, np.where(diff_pct>0.0, np.nan, -diff_pct), width=bin_widths, color='blue', label=rf'CTL $>$ {name_ehe}')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel(r'$\log_{10}\left<M_b\right>$')
    ax2.set_ylabel(f'Difference ({name_ehe} - CTL)')
    ax2.set_yscale('log')
    ax2.set_ylim(1.0e-4, 10.)
    ax2.legend()

    plt.show()

In [ ]:
def compare_csd_norm2(stat_ehe, stat_ctl, name_ehe, nbins, minmf, maxmf):
    bins = np.linspace(minmf, maxmf, nbins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_widths = np.diff(bins)

    # Use only positive, finite values before log10
    ehe_raw = np.asarray(stat_ehe[0], dtype=float)
    ctl_raw = np.asarray(stat_ctl[0], dtype=float)
    ehe_log = np.log10(ehe_raw[(ehe_raw > 0) & np.isfinite(ehe_raw)])
    ctl_log = np.log10(ctl_raw[(ctl_raw > 0) & np.isfinite(ctl_raw)])

    # Normalized histogram (fraction per bin)
    h_ehe, _ = np.histogram(ehe_log, bins=bins)
    h_ctl, _ = np.histogram(ctl_log, bins=bins)

    h_ehe = h_ehe / h_ehe.sum()
    h_ctl = h_ctl / h_ctl.sum()

    # Convert to %
    h_ehe_pct = 100.0 * h_ehe
    h_ctl_pct = 100.0 * h_ctl
    diff_pct = 100.0*(h_ehe_pct - h_ctl_pct)/h_ctl_pct
    print(diff_pct)

    fig = plt.figure(figsize=(12, 14))
    ax1 = fig.add_axes((0.1, 0.55, 0.85, 0.4))
    ax2 = fig.add_axes((0.1, 0.1, 0.85, 0.4))

    # Panel 1: normalized distributions
    ax1.bar(bin_centers, h_ctl_pct, width=bin_widths, color='black', alpha=0.45, label='CTL')
    ax1.bar(bin_centers, h_ehe_pct, width=bin_widths, color='red', alpha=0.45, label=name_ehe)
    ax1.axvline(np.median(ehe_log), color='red', linestyle='--')
    ax1.axvline(np.median(ctl_log), color='black', linestyle='--')
    ax1.set_yscale('log')
    ax1.set_ylabel('Cloud fraction per bin (%)')
    ax1.set_title('Normalized histograms')
    ax1.legend()

    ax2.bar(bin_centers, np.where(diff_pct>0.0, diff_pct, np.nan), width=bin_widths, color='red', label=rf'{name_ehe} $>$ CTL')
    ax2.bar(bin_centers, np.where(diff_pct>0.0, np.nan, -diff_pct), width=bin_widths, color='blue', label=rf'CTL $>$ {name_ehe}')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel(r'$\log_{10}\left<M_b\right>$')
    ax2.set_ylabel(f'Difference ({name_ehe} - CTL)')
    ax2.set_yscale('log')
    ax2.set_ylim(1.0e-2, 100.)
    ax2.legend()

    plt.show()

In [ ]:
nbins = 15 

In [ ]:
compare_csd_norm(stat_ehe18, stat_ctl, 'Smooth all', nbins, minmf, maxmf)

In [ ]:
compare_csd_norm2(stat_ehe18, stat_ctl, 'Smooth all', nbins, minmf, maxmf)

In [ ]:
compare_csd_norm(stat_ehe22, stat_ctl, 'Smooth lower', nbins, minmf, maxmf)

In [ ]:
compare_csd_norm2(stat_ehe22, stat_ctl, 'Smooth lower', nbins, minmf, maxmf)

In [ ]:
compare_csd_norm(stat_ehe21, stat_ctl, 'Smooth upper', nbins, minmf, maxmf)

In [ ]:
compare_csd_norm2(stat_ehe21, stat_ctl, 'Smooth upper', nbins, minmf, maxmf)